In [2]:
%%writefile .env
minha_chave=GKe752e73e675cc700c0eb72f3
secret_key=0d28e7f60c63d7c149ea5f4378530f91fca9ab2e57c6fbb970bcc12c98c33252

Overwriting .env


In [1]:
!pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# Carrega na memória as variáveis declaradas no arquivo oculto
load_dotenv()
access_key = os.getenv('minha_chave')
secret_key = os.getenv('secret_key')

# Instancia a sessão do Spark injetando dinamicamente os pacotes do Kafka e S3A AWS Hadoop
spark = SparkSession.builder \
    .appName("Security-DataLake-Pipeline") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.apache.hadoop:hadoop-aws:3.3.4") \
    .getOrCreate()

# Configuração detalhada do driver de mapeamento de objetos para o endpoint do Garage
sc = spark.sparkContext
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", access_key)
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", secret_key)
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://garage:3900")
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

# Parâmetros de compatibilidade regional do Garage
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint.region", "garage")
sc._jsc.hadoopConfiguration().set("fs.s3a.region", "garage")
sc._jsc.hadoopConfiguration().set("fs.s3a.change.detection.mode", "none")
sc._jsc.hadoopConfiguration().set("fs.s3a.change.detection.source", "none")

# ⚡ TUNING DE PERFORMANCE: ACELERAÇÃO DE ESCRITA NO OBJECT STORAGE LOCAL
sc._jsc.hadoopConfiguration().set("mapreduce.fileoutputcommitter.algorithm.version", "2")
sc._jsc.hadoopConfiguration().set("fs.s3a.directory.marker.retention", "keep")
sc._jsc.hadoopConfiguration().set("fs.s3a.fast.upload", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.fast.upload.buffer", "disk")

print("🚀 Sessão Spark tunada e otimizada para o Garage HQ!")

🚀 Sessão Spark tunada e otimizada para o Garage HQ!


In [ ]:
dados_teste = [
    ("VLAN_30", "192.168.30.99", "Mirai Port Scan Detectado", "ALTA"),
    ("VLAN_40", "192.168.40.12", "Tentativa Bruteforce SSH", "MEDIA")
]
colunas = ["origem_vlan", "ip_origem", "evento", "severidade"]
df_teste = spark.createDataFrame(dados_teste, schema=colunas)

try:
    df_teste.write.format("parquet").mode("overwrite").save("s3a://meu-data-lake/teste_alertas/")
    print("✨ Sucesso! O Spark conseguiu autenticar, gravar e fechar pacotes no Garage HQ.")
    spark.read.parquet("s3a://meu-data-lake/teste_alertas/").show()
except Exception as e:
    print(f"❌ Falha crítica de privilégio ou IO no Object Storage: {e}")

✨ Sucesso! O Spark conseguiu autenticar, gravar e fechar pacotes no Garage HQ.
+-----------+-------------+--------------------+----------+
|origem_vlan|    ip_origem|              evento|severidade|
+-----------+-------------+--------------------+----------+
|    VLAN_30|192.168.30.99|Mirai Port Scan D...|      ALTA|
|    VLAN_40|192.168.40.12|Tentativa Brutefo...|     MEDIA|
+-----------+-------------+--------------------+----------+



In [ ]:
from pyspark.sql.functions import col, from_json, current_timestamp, coalesce
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType
)

# Schema tolerante a GoFlow2 nativo e testes manuais
goflow_schema = StructType([
    StructField("Type", StringType(), True),
    StructField("TimeReceived", LongType(), True),
    StructField("SequenceNum", LongType(), True),
    StructField("SamplingRate", IntegerType(), True),
    StructField("FlowDirection", IntegerType(), True),

    StructField("Bytes", LongType(), True),
    StructField("Packets", LongType(), True),
    StructField("bytes", LongType(), True),
    StructField("packets", LongType(), True),

    StructField("SrcAddr", StringType(), True),
    StructField("DstAddr", StringType(), True),
    StructField("src_ip", StringType(), True),
    StructField("dst_ip", StringType(), True),

    StructField("SrcPort", IntegerType(), True),
    StructField("DstPort", IntegerType(), True),
    StructField("src_port", IntegerType(), True),
    StructField("dst_port", IntegerType(), True),

    StructField("Proto", IntegerType(), True),
    StructField("TcpFlags", IntegerType(), True),

    StructField("SrcVlan", IntegerType(), True),
    StructField("DstVlan", IntegerType(), True),
    StructField("VlanId", IntegerType(), True),
    StructField("vlan", IntegerType(), True),

    StructField("SamplerAddress", StringType(), True)
])

df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "ipfix-network-flow") \
    .option("startingOffsets", "earliest") \
    .load()

df_flows = (
    df_kafka
    .selectExpr(
        "CAST(key AS STRING) AS kafka_key",
        "CAST(value AS STRING) AS json_payload",
        "timestamp AS kafka_timestamp",
        "partition AS kafka_partition",
        "offset AS kafka_offset"
    )
    .select(
        "*",
        from_json(col("json_payload"), goflow_schema).alias("flow")
    )
    .select(
        col("flow.Type").alias("flow_type"),
        col("flow.TimeReceived").alias("time_received"),
        col("flow.SequenceNum").alias("sequence_num"),
        col("flow.SamplingRate").alias("sampling_rate"),
        col("flow.FlowDirection").alias("flow_direction"),

        coalesce(col("flow.SrcAddr"), col("flow.src_ip")).alias("src_ip"),
        coalesce(col("flow.DstAddr"), col("flow.dst_ip")).alias("dst_ip"),
        coalesce(col("flow.SrcPort"), col("flow.src_port")).alias("src_port"),
        coalesce(col("flow.DstPort"), col("flow.dst_port")).alias("dst_port"),

        col("flow.Proto").alias("protocol"),
        col("flow.TcpFlags").alias("tcp_flags"),

        coalesce(col("flow.Bytes"), col("flow.bytes")).alias("bytes"),
        coalesce(col("flow.Packets"), col("flow.packets")).alias("packets"),

        coalesce(
            col("flow.vlan"),
            col("flow.VlanId"),
            col("flow.SrcVlan"),
            col("flow.DstVlan")
        ).alias("vlan"),

        col("flow.SamplerAddress").alias("sampler_address"),

        col("kafka_timestamp"),
        col("kafka_partition"),
        col("kafka_offset"),

        current_timestamp().alias("ingestion_time")
    )
    .filter(col("src_ip").isNotNull())
)

# 🟢 CORREÇÃO CRÍTICA: Inicialização da gravação em streaming
query = df_flows.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "s3a://meu-data-lake/live_network_logs/") \
    .option("checkpointLocation", "s3a://meu-data-lake/checkpoints/goflow_pipeline/") \
    .partitionBy("vlan") \
    .start()

print("🔥 Pipeline GoFlow2 ativado e gravando em streaming no Garage HQ!")

+-----------+-------------+--------------------+----------+
|origem_vlan|    ip_origem|              evento|severidade|
+-----------+-------------+--------------------+----------+
|    VLAN_30|192.168.30.99|Mirai Port Scan D...|      ALTA|
|    VLAN_40|192.168.40.12|Tentativa Brutefo...|     MEDIA|
+-----------+-------------+--------------------+----------+



In [ ]:
import json

# 1. Verifica se a thread continua ativa em segundo plano
print(f"O streaming está ativo? {query.isActive}")

# 2. Exibe o progresso do último micro-batch
if query.lastProgress:
    print(json.dumps(query.lastProgress, indent=2))
else:
    print("Aguardando entrada de dados para exibir telemetria...")

🔥 Pipeline de streaming em tempo real ativado em segundo plano!


In [ ]:
# Exibe a tabela unificada contendo todos os registros persistidos
spark.read.parquet("s3a://meu-data-lake/live_network_logs/").show(truncate=False)

O streaming está ativo? True
null
